# Template Kosongan: Import → EDA → Data Cleaning → Export (Marimo Style)

> **Catatan Penting:** Notebook **Marimo** aslinya berformat `.py` (`marimo edit nama_file.py`). 
> File `.ipynb` ini disusun sebagai **template kosongan terstruktur** yang siap dibuka di Jupyter/VS Code **atau** dikonversi langsung ke format Marimo dengan perintah:
> ```bash
> marimo convert template_eda_cleaning_export_marimo.ipynb -o template_eda_cleaning_export_marimo.py
> ```

## 0. Setup & Import Library

In [1]:
import marimo as mo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## 1. Import Data (Kosongan — Isi Path Dataset Anda)

In [2]:
# Isi path file sumber Anda di bawah ini (CSV, Excel, JSON, atau Parquet)
INPUT_PATH = Path(r"C:\01.Training\01.Datasheets\online_shoppers_intention.csv")  # Contoh: Path(r"C:\data\dataset.csv")
CSV_DELIMITER = None    # None = auto-detect, atau isi "," / ";"
CSV_ENCODING = "utf-8-sig"
EXCEL_SHEET = 0

def load_dataset(path: Path) -> pd.DataFrame:
    if not str(path) or not path.is_file():
        print("Status: Path belum diisi atau file tidak ditemukan.")
        return pd.DataFrame()
    ext = path.suffix.lower()
    if ext == ".csv":
        return pd.read_csv(path, sep=CSV_DELIMITER, engine="python", encoding=CSV_ENCODING)
    if ext in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=EXCEL_SHEET)
    if ext == ".json":
        return pd.read_json(path)
    if ext == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Format file tidak didukung: {ext}")

df_raw = load_dataset(INPUT_PATH)
print(f"Dimensi df_raw: {df_raw.shape}")

Dimensi df_raw: (12330, 18)


## 2. Exploratory Data Analysis (EDA) Awal

In [3]:
def audit_eda(df: pd.DataFrame) -> None:
    if df.empty:
        print("Dataset masih kosong. Muat data pada tahap 1 terlebih dahulu.")
        return
    print("=== RINGKASAN UMUM ===")
    print(f"Baris: {len(df):,} | Kolom: {df.shape[1]} | Duplikat: {df.duplicated().sum():,} | Total Missing: {df.isna().sum().sum():,}")
    print("\n=== PROFIL KOLOM ===")
    profile = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=True),
    })
    display(profile)
    print("\n=== STATISTIK DESKRIPTIF ===")
    display(df.describe(include="all").T)

audit_eda(df_raw)

=== RINGKASAN UMUM ===
Baris: 12,330 | Kolom: 18 | Duplikat: 125 | Total Missing: 0

=== PROFIL KOLOM ===


,dtype,missing,missing_%,unique
Administrative,int64,0,0.0,27
Administrative_Duration,float64,0,0.0,3335
Informational,int64,0,0.0,17
Informational_Duration,float64,0,0.0,1258
ProductRelated,int64,0,0.0,311
ProductRelated_Duration,float64,0,0.0,9551
BounceRates,float64,0,0.0,1872
ExitRates,float64,0,0.0,4777
PageValues,float64,0,0.0,2704
SpecialDay,float64,0,0.0,6



=== STATISTIK DESKRIPTIF ===


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Administrative,12330.0,NaN,NaN,NaN,2.315166,3.321784,0.0,0.0,1.0,4.0,27.0
Administrative_Duration,12330.0,NaN,NaN,NaN,80.818611,176.779107,0.0,0.0,7.5,93.25625,3398.75
Informational,12330.0,NaN,NaN,NaN,0.503569,1.270156,0.0,0.0,0.0,0.0,24.0
Informational_Duration,12330.0,NaN,NaN,NaN,34.472398,140.749294,0.0,0.0,0.0,0.0,2549.375
ProductRelated,12330.0,NaN,NaN,NaN,31.731468,44.475503,0.0,7.0,18.0,38.0,705.0
ProductRelated_Duration,12330.0,NaN,NaN,NaN,1194.74622,1913.669288,0.0,184.1375,598.936905,1464.157214,63973.52223
BounceRates,12330.0,NaN,NaN,NaN,0.022191,0.048488,0.0,0.0,0.003112,0.016813,0.2
ExitRates,12330.0,NaN,NaN,NaN,0.043073,0.048597,0.0,0.014286,0.025156,0.05,0.2
PageValues,12330.0,NaN,NaN,NaN,5.889258,18.568437,0.0,0.0,0.0,0.0,361.763742
SpecialDay,12330.0,NaN,NaN,NaN,0.061427,0.198917,0.0,0.0,0.0,0.0,1.0


In [8]:
import numpy as np
import pandas as pd

def audit_eda(df: pd.DataFrame) -> None:
    if df.empty:
        print("Dataset masih kosong. Muat data pada tahap 1 terlebih dahulu.")
        return

    num_cols = df.select_dtypes(include="number").columns
    if len(num_cols) > 0:
        print("\n=== IQR & OUTLIER ===")
        iqr_rows = []
        for col in num_cols:
            s = df[col].dropna()
            if s.empty:
                iqr_rows.append({
                    "kolom": col,
                    "q1": np.nan,
                    "q3": np.nan,
                    "iqr": np.nan,
                    "lower_bound": np.nan,
                    "upper_bound": np.nan,
                    "outlier_low": 0,
                    "outlier_high": 0,
                    "outlier_count": 0,
                    "outlier_%": 0.0,
                })
                continue

            q1 = s.quantile(0.25)
            q3 = s.quantile(0.75)
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            low_mask = s < lower_bound
            high_mask = s > upper_bound
            outlier_count = int((low_mask | high_mask).sum())
            outlier_pct = round((outlier_count / len(s)) * 100, 2)

            iqr_rows.append({
                "kolom": col,
                "q1": round(q1, 4),
                "q3": round(q3, 4),
                "iqr": round(iqr, 4),
                "lower_bound": round(lower_bound, 4),
                "upper_bound": round(upper_bound, 4),
                "outlier_low": int(low_mask.sum()),
                "outlier_high": int(high_mask.sum()),
                "outlier_count": outlier_count,
                "outlier_%": outlier_pct,
            })

        iqr_df = pd.DataFrame(iqr_rows).set_index("kolom")
        display(iqr_df)

        outlier_cols_df = iqr_df[iqr_df["outlier_count"] > 0]
        print("\n=== KOLOM YANG MEMILIKI OUTLIER ===")
        if outlier_cols_df.empty:
            print("Tidak ada kolom numerik yang memiliki outlier berdasarkan metode IQR.")
        else:
            print(f"Daftar kolom ({len(outlier_cols_df)}): {', '.join(outlier_cols_df.index)}")
            display(outlier_cols_df[["lower_bound", "upper_bound", "outlier_low", "outlier_high", "outlier_count", "outlier_%"]])
    else:
        print("\n=== IQR & OUTLIER ===")
        print("Tidak ada kolom numerik untuk dihitung IQR dan outlier.")

audit_eda(df_raw)


=== IQR & OUTLIER ===


,q1,q3,iqr,lower_bound,upper_bound,outlier_low,outlier_high,outlier_count,outlier_%
kolom,,,,,,,,,
Administrative,0.0000,4.0000,4.0000,-6.0000,10.0000,0,404,404,3.28
Administrative_Duration,0.0000,93.2562,93.2562,-139.8844,233.1406,0,1172,1172,9.51
Informational,0.0000,0.0000,0.0000,0.0000,0.0000,0,2631,2631,21.34
Informational_Duration,0.0000,0.0000,0.0000,0.0000,0.0000,0,2405,2405,19.51
ProductRelated,7.0000,38.0000,31.0000,-39.5000,84.5000,0,987,987,8.00
ProductRelated_Duration,184.1375,1464.1572,1280.0197,-1735.8921,3384.1868,0,961,961,7.79
BounceRates,0.0000,0.0168,0.0168,-0.0252,0.0420,0,1551,1551,12.58
ExitRates,0.0143,0.0500,0.0357,-0.0393,0.1036,0,1099,1099,8.91
PageValues,0.0000,0.0000,0.0000,0.0000,0.0000,0,2730,2730,22.14



=== KOLOM YANG MEMILIKI OUTLIER ===
Daftar kolom (14): Administrative, Administrative_Duration, Informational, Informational_Duration, ProductRelated, ProductRelated_Duration, BounceRates, ExitRates, PageValues, SpecialDay, OperatingSystems, Browser, Region, TrafficType


,lower_bound,upper_bound,outlier_low,outlier_high,outlier_count,outlier_%
kolom,,,,,,
Administrative,-6.0000,10.0000,0,404,404,3.28
Administrative_Duration,-139.8844,233.1406,0,1172,1172,9.51
Informational,0.0000,0.0000,0,2631,2631,21.34
Informational_Duration,0.0000,0.0000,0,2405,2405,19.51
ProductRelated,-39.5000,84.5000,0,987,987,8.00
ProductRelated_Duration,-1735.8921,3384.1868,0,961,961,7.79
BounceRates,-0.0252,0.0420,0,1551,1551,12.58
ExitRates,-0.0393,0.1036,0,1099,1099,8.91
PageValues,0.0000,0.0000,0,2730,2730,22.14


In [ ]:
import pandas as pd

num_cols = df_raw.select_dtypes(include="number").columns
mask = pd.Series(False, index=df_raw.index)
info = pd.Series("", index=df_raw.index)

for col in num_cols:
    q1, q3 = df_raw[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    col_mask = (df_raw[col] < low) | (df_raw[col] > high)
    mask |= col_mask

    for idx in df_raw.index[col_mask]:
        val = df_raw.loc[idx, col]
        arah = "low" if val < low else "high"
        teks = f"{col}={val} ({arah})"
        info.loc[idx] = f"{info.loc[idx]} | {teks}".lstrip(" | ")

outlier_rows = df_raw[mask].copy()
outlier_rows.insert(0, "outlier_detail", info[mask])

outlier_rows.to_excel("outlier_rows", index=True)
print(f"Tersimpan ke outlier_rows.xlsx ({len(outlier_rows):,} baris)")

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (4269600664.py, line 23)

In [ ]:
# Area kosong eksplorasi & visualisasi kustom (isi nama kolom setelah data dimuat)
KOLOM_TARGET = ""   # Contoh: "Revenue"
KOLOM_ANALISIS = "" # Contoh: "ProductRelated_Duration"

# if not df_raw.empty and KOLOM_ANALISIS in df_raw.columns:
#     df_raw[KOLOM_ANALISIS].hist(bins=30)
#     plt.title(f"Distribusi {KOLOM_ANALISIS}")
#     plt.show()

## 3. Data Cleaning Terkendali (Salinan di Memori)

In [ ]:
# Konfigurasi aturan cleaning (default False / aman tanpa mengubah data mentah)
TRIM_WHITESPACE = False
DROP_DUPLICATES = False
MISSING_STRATEGY = "none"  # Pilihan: "none", "drop_rows", "impute_median_mode"
COLUMNS_TO_DROP = []       # Contoh: ["kolom_tidak_terpakai"]

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    cleaned = df.copy(deep=True)
    if TRIM_WHITESPACE:
        for col in cleaned.select_dtypes(include=["object", "string"]).columns:
            cleaned[col] = cleaned[col].map(lambda x: x.strip() if isinstance(x, str) else x)
    if DROP_DUPLICATES:
        cleaned = cleaned.drop_duplicates().copy()
    if MISSING_STRATEGY == "drop_rows":
        cleaned = cleaned.dropna().copy()
    elif MISSING_STRATEGY == "impute_median_mode":
        for col in cleaned.columns:
            if cleaned[col].isna().any():
                if pd.api.types.is_numeric_dtype(cleaned[col]) and not pd.api.types.is_bool_dtype(cleaned[col]):
                    cleaned[col] = cleaned[col].fillna(cleaned[col].median())
                else:
                    mode_vals = cleaned[col].mode(dropna=True)
                    if not mode_vals.empty:
                        cleaned[col] = cleaned[col].fillna(mode_vals.iloc[0])
    valid_drop = [c for c in COLUMNS_TO_DROP if c in cleaned.columns]
    if valid_drop:
        cleaned = cleaned.drop(columns=valid_drop)
    return cleaned

df_clean = clean_data(df_raw)
print(f"Dimensi sebelum cleaning: {df_raw.shape} -> sesudah: {df_clean.shape}")

## 4. Validasi Pasca-Cleaning

In [ ]:
# Cek ulang apakah masih ada missing atau duplikat setelah aturan cleaning diterapkan
if not df_clean.empty:
    print("Duplikat tersisa :", int(df_clean.duplicated().sum()))
    print("Missing tersisa  :", int(df_clean.isna().sum().sum()))
    display(df_clean.head(5))
else:
    print("df_clean masih kosong.")

## 5. Export Data Clean (Kosongan — Isi Path & Format Output)

In [ ]:
# Konfigurasi export hasil bersih (tidak menimpa file sumber)
EXPORT_ENABLED = False
OUTPUT_PATH = Path(r"C:\01.Training\dataclean")  # Bisa folder atau nama file lengkap
EXPORT_FORMAT = "csv"       # Pilihan: "csv", "xlsx", "json", "parquet"

def export_clean_data(df: pd.DataFrame, out_path: Path, fmt: str, enabled: bool = False) -> None:
    if not enabled:
        print("Export belum diaktifkan (EXPORT_ENABLED = False).")
        return
    if df.empty:
        print("Gagal export: df_clean masih kosong.")
        return
    out_path = Path(out_path)
    if str(out_path) in {"", "."}:
        print("Gagal export: OUTPUT_PATH belum diisi.")
        return
    fmt_clean = fmt.strip().lower()
    extensions = {"csv": ".csv", "xlsx": ".xlsx", "json": ".json", "parquet": ".parquet"}
    if fmt_clean not in extensions:
        raise ValueError(f"Format export tidak didukung: {fmt}")
    # Jika OUTPUT_PATH berupa folder (atau belum memiliki ekstensi), buat nama file otomatis.
    if out_path.is_dir() or not out_path.suffix:
        out_path = out_path / f"data_cleaned{extensions[fmt_clean]}"
    if str(INPUT_PATH) and INPUT_PATH.exists() and INPUT_PATH.resolve() == out_path.resolve():
        raise ValueError("OUTPUT_PATH tidak boleh sama dengan INPUT_PATH.")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    if fmt_clean == "csv":
        df.to_csv(out_path, index=False, encoding="utf-8-sig")
    elif fmt_clean == "xlsx":
        df.to_excel(out_path, index=False)
    elif fmt_clean == "json":
        df.to_json(out_path, orient="records", indent=2, force_ascii=False)
    elif fmt_clean == "parquet":
        df.to_parquet(out_path, index=False)
    print(f"Export berhasil: {out_path} ({len(df):,} baris x {df.shape[1]} kolom, format={fmt_clean})")

export_clean_data(df_clean, OUTPUT_PATH, EXPORT_FORMAT, enabled=EXPORT_ENABLED)